In [1]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate

# Load environment variables securely
current_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(current_dir, "..", ".."))
env_path = os.path.join(project_root, ".env")
load_dotenv(dotenv_path=env_path, override=True)

# Initialize the Gemini model
model = ChatGoogleGenerativeAI(
    model="gemini-flash-lite-latest",
    temperature=0.1
)

# ==========================================
# The Old, Manual Way (Nested)
# ==========================================
prompt = ChatPromptTemplate.from_template("درباره {topic} یک جمله بگو.")

messages = prompt.invoke({"topic": "AI"})
response = model.invoke(messages)

print(response.content)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


[{'type': 'text', 'text': 'هوش مصنوعی (AI) فناوری شگفت\u200cانگیزی است که به ماشین\u200cها امکان می\u200cدهد مانند انسان\u200cها فکر کنند، یاد بگیرند و مسائل را حل کنند.', 'extras': {'signature': 'EmAKXgFpFH0TfuDlnq2+r9aAV8+K89Dh+zRnP27cMEy5fiZ+CIgMRj4hpOZyQzlgtKFBF2ahlz2QD4vezf++fl6xeJFYXhghYg4Q5Lt6pS1uGX5mFH6ZqzDF3vAqaoGh7dE='}}]


In [2]:
from langchain_core.output_parsers import StrOutputParser

# 1. Define the Prompt Template
prompt = ChatPromptTemplate.from_messages([
    ("system", "تو یک مترجم حرفه‌ای هستی. متن کاربر را به {language} ترجمه کن."),
    ("human", "{text}")
])

# 2. Define the Parser (Converts AI Message object to plain text/string)
parser = StrOutputParser()

# 3. Create the Chain using LCEL (Pipe syntax)
# Flow: Prompt -> Model -> Parser
chain = prompt | model | parser

# 4. Execute the Chain
result = chain.invoke({
    "language": "فرانسوی", 
    "text": "سلام دنیا"
})

print(result)

Bonjour le monde


In [3]:
from langchain_core.output_parsers import CommaSeparatedListOutputParser

# 1. Define a new template for extracting keywords
tag_prompt = ChatPromptTemplate.from_template(
    "۳ کلمه کلیدی مهم متن زیر را استخراج کن و با ویرگول جدا کن:\n{text}"
)

# 2. Define a new parser (Takes text, splits by commas, returns a Python List)
list_parser = CommaSeparatedListOutputParser()

# 3. The structure remains the same: Prompt | Model | Parser
tag_chain = tag_prompt | model | list_parser

# 4. Execute the chain
result = tag_chain.invoke({
    "text": "هوش مصنوعی مولد، پارادایم توسعه نرم‌افزار را تغییر داده است."
})

print(f"Output: {result}")
print(f"Data Type: {type(result)}")

Output: ['هوش مصنوعی مولد', 'توسعه نرم\u200cافزار', 'پارادایم']
Data Type: <class 'list'>
